In [ ]:
import kagglehub
google_gemma_4_transformers_gemma_4_e4b_it_1_path = kagglehub.model_download('google/gemma-4/Transformers/gemma-4-e4b-it/1')
google_gemma_4_transformers_gemma_4_e2b_it_1_path = kagglehub.model_download('google/gemma-4/Transformers/gemma-4-e2b-it/1')
print('Data source import complete.')

# Memory Anchor — Multimodal RAG for Dementia Care

**Gemma 4 for Good Hackathon Submission**

An offline, privacy-first app that helps dementia patients remember loved ones using **Multimodal RAG** (Retrieval-Augmented Generation) instead of fine-tuning.

### Why RAG instead of Fine-Tuning?
| | Fine-Tuning | Multimodal RAG |
|---|---|---|
| Training needed | Yes (GPU hours) | No |
| Add new memories | Retrain required | Instant |
| Hallucination risk | Higher | Lower (grounded in retrieved data) |
| Hardware needs | GPU for training | CPU-friendly |
| Dataset size | Needs 200-500+ examples | Works with 1 photo per person |

### Architecture
```
User: "Who is this?" + [photo]
  → 1. CLIP encodes query image → embedding
  → 2. ChromaDB similarity search → top-3 matching family photos
  → 3. Retrieve: name, relationship, story, voice clip path
  → 4. Gemma generates warm, grounded response
  → 5. Play voice clip if available
```

## Step 0: Install Dependencies

In [ ]:
import torch; print(f'Pre-installed PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')

# Install deps WITHOUT torch (keep Kaggle's pre-installed CUDA-compatible version)
!pip install -q chromadb Pillow accelerate bitsandbytes loguru tqdm pyyaml numpy
!pip install -q sentence-transformers --no-deps
!pip install -q "huggingface-hub>=1.5.0"
# Transformers from source for Gemma 4 support
!pip install -q --no-deps git+https://github.com/huggingface/transformers.git

import importlib; importlib.invalidate_caches()
import torch; print(f'Post-install PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, Compute: {torch.cuda.get_device_capability(0)}')
print('Dependencies installed.')


## Step 1: Generate Mock Family Data
In real use, these would be actual family photos and voice clips.

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import numpy as np

MOCK_FAMILY = [
    {"name": "sarah", "person_name": "Sarah", "relationship": "daughter",
     "captions": ["Sarah smiling at her college graduation, May 2018",
                  "Sarah baking cookies in the kitchen, Christmas 2019",
                  "Sarah and Mom at the beach, summer 2020"],
     "story": "Sarah is your youngest daughter. She loves baking \u2014 especially those chocolate chip cookies you two always made together. She graduated from State University in 2018 with a degree in nursing. She calls you every Sunday evening.",
     "color": (255, 182, 193)},
    {"name": "arki", "person_name": "Arki", "relationship": "son",
     "captions": ["Arki building the birdhouse in the backyard, spring 1998",
                  "Arki at his wedding, June 2015",
                  "Arki holding baby Maya, his first child, 2020"],
     "story": "Arki is your eldest son. He built you that beautiful birdhouse in the backyard back in '98 \u2014 it's still there! He married Lisa in 2015 and they have a daughter named Maya. He works as a carpenter, just like his grandfather.",
     "color": (173, 216, 230)},
    {"name": "maya", "person_name": "Maya", "relationship": "granddaughter",
     "captions": ["Maya's first birthday party, 2021",
                  "Maya drawing pictures with Grandma, 2023",
                  "Maya at the park with her teddy bear, 2024"],
     "story": "Maya is your granddaughter \u2014 Arki and Lisa's little girl. She was born in 2020. She loves drawing and always makes pictures for you. She calls you 'Nana'.",
     "color": (255, 255, 200)},
    {"name": "robert", "person_name": "Robert", "relationship": "husband",
     "captions": ["Robert and you on your wedding day, 1975",
                  "Robert fishing at the lake, summer 1990",
                  "Robert in the garden with his roses, 2010"],
     "story": "Robert is your husband. You married in 1975. He loved fishing and tending his rose garden. His favorite song was 'Moon River'. He always said you were the best thing that ever happened to him.",
     "color": (200, 230, 200)},
    {"name": "margaret", "person_name": "Margaret", "relationship": "best friend",
     "captions": ["Margaret and you at the quilting circle, 2018",
                  "Margaret's 75th birthday party, 2022"],
     "story": "Margaret is your best friend since high school \u2014 over 55 years! You two go to the quilting circle every Thursday at the community center. She lives three houses down on Maple Street.",
     "color": (230, 200, 230)},
    {"name": "buddy", "person_name": "Buddy", "relationship": "pet dog",
     "captions": ["Buddy as a puppy, the day you brought him home, 2019",
                  "Buddy playing in the yard with Maya, 2023"],
     "story": "Buddy is your golden retriever. You got him as a puppy in 2019. He loves belly rubs and follows you everywhere. He sleeps at the foot of your bed every night.",
     "color": (255, 215, 140)},
    {"name": "dr_chen", "person_name": "Dr. Chen", "relationship": "doctor",
     "captions": ["Dr. Chen at the clinic, your Tuesday checkups"],
     "story": "Dr. Chen is your doctor. You see her every Tuesday afternoon at 2pm at the Riverside Clinic. She's been your doctor for 5 years and is very kind.",
     "color": (220, 220, 240)},
    {"name": "lisa", "person_name": "Lisa", "relationship": "daughter-in-law",
     "captions": ["Lisa and Arki's wedding day, June 2015",
                  "Lisa teaching Maya to ride a bike, 2023"],
     "story": "Lisa is Arki's wife \u2014 your daughter-in-law. She's a schoolteacher and always brings your favorite lemon cake when she visits.",
     "color": (255, 218, 185)},
]

def generate_mock_photo(person, index, output_path):
    width, height = 512, 512
    img = Image.new("RGB", (width, height), person["color"])
    draw = ImageDraw.Draw(img)
    cx, cy = width // 2, height // 2 - 30
    draw.ellipse([cx-80, cy-80, cx+80, cy+80], fill="white", outline="gray", width=2)
    draw.ellipse([cx-30, cy-20, cx-10, cy], fill="gray")
    draw.ellipse([cx+10, cy-20, cx+30, cy], fill="gray")
    draw.arc([cx-30, cy+10, cx+30, cy+40], 0, 180, fill="gray", width=2)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 36)
        sfont = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
    except (IOError, OSError):
        font = ImageFont.load_default()
        sfont = font
    draw.text((cx, height-100), person["person_name"], fill="black", anchor="mm", font=font)
    caption = person["captions"][index] if index < len(person["captions"]) else ""
    if caption:
        draw.text((cx, height-60), caption[:45], fill="dimgray", anchor="mm", font=sfont)
    filepath = output_path / f"photo_{index+1}.jpg"
    img.save(filepath, quality=90)
    return filepath

data_dir = Path("./data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

for person in MOCK_FAMILY:
    person_dir = data_dir / person["name"]
    person_dir.mkdir(parents=True, exist_ok=True)
    for i in range(len(person["captions"])):
        generate_mock_photo(person, i, person_dir)
    (person_dir / "caption.txt").write_text("\n".join(person["captions"]))
    (person_dir / "story.txt").write_text(person["story"])
    (person_dir / "meta.json").write_text(json.dumps({"relationship": person["relationship"]}, indent=2))
    print(f"  {person['person_name']} ({person['relationship']}): {len(person['captions'])} photos")

print(f"\nGenerated {len(MOCK_FAMILY)} family members")


In [ ]:
import os, glob

print("=== Checking /kaggle/input/ for Gemma 4 ===")
if os.path.exists("/kaggle/input/"):
    print("\n/kaggle/input/ contents:")
    for item in os.listdir("/kaggle/input/"):
        print(f"  - {item}")
        item_path = f"/kaggle/input/{item}"
        if os.path.isdir(item_path):
            try:
                sub = os.listdir(item_path)[:5]
                print(f"    -> {sub}")
            except: pass

print("\n=== Searching for Gemma model directories ===")
for pattern in ["/kaggle/input/**/gemma-4-e*b-it/1", "/kaggle/input/**/gemma-4*/1"]:
    for p in glob.glob(pattern, recursive=True):
        print(f"  Found: {p}")
        print(f"    Files: {os.listdir(p)[:8]}")


## Step 2: Initialize Embedding Models
**Force CPU for CLIP/text models** — GPU reserved for Gemma LLM.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_DEVICE = "cpu"
print(f"Embedding device: {EMBED_DEVICE}")
print(f"GPU available: {torch.cuda.is_available()} (reserved for Gemma LLM only)")

clip_model = SentenceTransformer("clip-ViT-B-32", device=EMBED_DEVICE)
print(f"CLIP loaded: clip-ViT-B-32 on {EMBED_DEVICE}")

text_model = SentenceTransformer("all-MiniLM-L6-v2", device=EMBED_DEVICE)
print(f"Text model loaded: all-MiniLM-L6-v2 on {EMBED_DEVICE}")


## Step 3: Build the Vector Store (ChromaDB)

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path="./data/embeddings",
    settings=Settings(anonymized_telemetry=False),
)
image_collection = chroma_client.get_or_create_collection(name="family_images", metadata={"hnsw:space": "cosine"})
text_collection = chroma_client.get_or_create_collection(name="family_text", metadata={"hnsw:space": "cosine"})
print(f"ChromaDB initialized (local only, privacy-first)")
print(f"  Image collection: {image_collection.count()} entries")
print(f"  Text collection:  {text_collection.count()} entries")


## Step 4: Ingest Family Data into Vector Store

In [ ]:
from PIL import Image

data_dir = Path("./data/raw")
total_images = 0
total_texts = 0

for person_dir in sorted(data_dir.iterdir()):
    if not person_dir.is_dir():
        continue
    person_name = person_dir.name.replace("_", " ").title()
    meta_file = person_dir / "meta.json"
    relationship = ""
    if meta_file.exists():
        with open(meta_file) as f:
            relationship = json.load(f).get("relationship", "")
    caption_file = person_dir / "caption.txt"
    captions = caption_file.read_text().splitlines() if caption_file.exists() else []
    story_file = person_dir / "story.txt"
    story = story_file.read_text().strip() if story_file.exists() else ""
    image_files = sorted(f for f in person_dir.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))

    for i, img_path in enumerate(image_files):
        caption = captions[i].strip() if i < len(captions) else ""
        img = Image.open(img_path).convert("RGB")
        img_embedding = clip_model.encode(img, convert_to_numpy=True)
        img_meta = {
            "person_name": person_name,
            "relationship": relationship,
            "caption": caption,
            "story": story,
            "image_path": str(img_path.resolve()),
            "audio_path": "",
        }
        image_collection.upsert(ids=[f"{person_dir.name}_img_{i}"], embeddings=[img_embedding.tolist()], metadatas=[img_meta])
        total_images += 1
        combined_text = f"{person_name}. {caption}. {story}".strip()
        if combined_text:
            text_embedding = text_model.encode(combined_text, convert_to_numpy=True)
            text_collection.upsert(ids=[f"{person_dir.name}_txt_{i}"], embeddings=[text_embedding.tolist()], metadatas=[img_meta])
            total_texts += 1
    print(f"  {person_name} ({relationship}): {len(image_files)} images indexed")

print(f"\nTotal indexed: {total_images} images, {total_texts} text memories")


## Step 5: Load Gemma for Response Generation
No fine-tuning needed — Gemma is used as-is with retrieved context.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os, glob

# Check if GPU is usable (P100 compute 6.0 is NOT compatible with PyTorch 2.x CUDA kernels)
USE_GPU = False
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU: {gpu_name}, Compute capability: {cap}')
    if cap[0] >= 7:  # T4=7.5, V100=7.0, A100=8.0 — all fine. P100=6.0 — broken.
        USE_GPU = True
        print('GPU compatible — will use GPU for Gemma')
    else:
        print(f'GPU compute {cap} < 7.0 — incompatible with PyTorch {torch.__version__}')
        print('Falling back to CPU (using smaller E2B model)')
else:
    print('No GPU — using CPU')

# Prefer E4B on GPU, E2B on CPU
PREFERRED = ['e4b', 'e2b'] if USE_GPU else ['e2b', 'e4b']

MODEL_NAME = None
for variant in PREFERRED:
    for p in glob.glob(f'/kaggle/input/**/gemma-4-{variant}-it/1', recursive=True):
        if os.path.exists(os.path.join(p, 'config.json')):
            MODEL_NAME = p
            break
    if MODEL_NAME:
        break

if not MODEL_NAME:
    import kagglehub
    for variant in PREFERRED:
        try:
            MODEL_NAME = kagglehub.model_download(f'google/gemma-4/Transformers/gemma-4-{variant}-it/1')
            break
        except: pass

print(f'Model: {MODEL_NAME}')
print(f'Files: {os.listdir(MODEL_NAME)[:8]}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if USE_GPU:
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True
    )
    print('Gemma loaded on GPU (float16)')
else:
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='cpu',
        trust_remote_code=True
    )
    print('Gemma loaded on CPU (bfloat16)')

print(f'Parameters: {sum(p.numel() for p in llm.parameters()):,}')
print(f'Device: {next(llm.parameters()).device}')


## Step 6: Define the RAG Pipeline

In [ ]:
SYSTEM_PROMPT = """You are Memory Anchor, a warm and patient companion for someone with dementia. Your job is to help them remember their loved ones.

RULES:
- ONLY use the information provided in the RETRIEVED MEMORIES below.
- NEVER invent facts, names, dates, or stories that are not in the memories.
- If you don't recognise someone, say so gently.
- Speak simply, warmly, and with patience.
- Use the person's name early in your response.
- Reference specific shared memories to spark recognition."""

def retrieve_by_image(query_image, top_k=3):
    if isinstance(query_image, (str, Path)):
        query_image = Image.open(query_image).convert("RGB")
    embedding = clip_model.encode(query_image, convert_to_numpy=True)
    return image_collection.query(query_embeddings=[embedding.tolist()], n_results=top_k)

def retrieve_by_text(query, top_k=3):
    embedding = text_model.encode(query, convert_to_numpy=True)
    return text_collection.query(query_embeddings=[embedding.tolist()], n_results=top_k)

def format_context(results):
    if not results["ids"][0]:
        return "No matching memories found."
    parts = []
    for i, (doc_id, meta, dist) in enumerate(zip(results["ids"][0], results["metadatas"][0], results["distances"][0]), 1):
        confidence = max(0, 1 - dist)
        parts.append(f"Memory {i} (confidence: {confidence:.0%}):\n  Name: {meta['person_name']}\n  Relationship: {meta.get('relationship', 'Unknown')}\n  Caption: {meta.get('caption', 'No caption')}\n  Story: {meta.get('story', 'No story available')}")
    return "\n\n".join(parts)

def generate_response(context, question):
    prompt = f"RETRIEVED MEMORIES:\n{context}\n\nUSER'S QUESTION: {question}\n\nRespond warmly. Ground every fact in the retrieved memories above."
    messages = [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + prompt}]
    
    encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(llm.device)
        attention_mask = None
    else:
        # BatchEncoding or dict-like
        input_ids = encoded['input_ids'].to(llm.device)
        attention_mask = encoded.get('attention_mask')
        if attention_mask is not None:
            attention_mask = attention_mask.to(llm.device)
    
    gen_kwargs = dict(max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
    if attention_mask is not None:
        gen_kwargs['attention_mask'] = attention_mask
    
    with torch.no_grad():
        output = llm.generate(input_ids=input_ids, **gen_kwargs)
    return tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_about_photo(image_path, question="Who is this person?"):
    results = retrieve_by_image(image_path)
    context = format_context(results)
    response = generate_response(context, question)
    return response, results

def ask_about_person(question):
    results = retrieve_by_text(question)
    context = format_context(results)
    response = generate_response(context, question)
    return response, results

print("RAG pipeline ready!")


## Step 7: Test — Photo Queries

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("TEST 1: Showing Sarah's graduation photo")
print("=" * 60)
test_img = "./data/raw/sarah/photo_1.jpg"
response, results = ask_about_photo(test_img, "Who is this person? Tell me about her.")
img = Image.open(test_img)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title("Query Image: Sarah")
plt.axis("off")
plt.show()
print(f"\nMemory Anchor says:\n{response}")
print(f"\nTop match: {results['metadatas'][0][0]['person_name']} (distance: {results['distances'][0][0]:.4f})")

print("\n" + "=" * 60)
print("TEST 2: Showing Arki's birdhouse photo")
print("=" * 60)
test_img = "./data/raw/arki/photo_1.jpg"
response, results = ask_about_photo(test_img, "Who is this? What did he build for me?")
img = Image.open(test_img)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title("Query Image: Arki")
plt.axis("off")
plt.show()
print(f"\nMemory Anchor says:\n{response}")


## Step 8: Test — Text Queries

In [ ]:
print("=" * 60)
print("TEST 3: 'Who bakes cookies with me?'")
print("=" * 60)
response, results = ask_about_person("Who bakes cookies with me?")
print(f"\nMemory Anchor says:\n{response}")
print(f"Top match: {results['metadatas'][0][0]['person_name']}")

print("\n" + "=" * 60)
print("TEST 4: 'Tell me about my dog'")
print("=" * 60)
response, results = ask_about_person("Tell me about my dog")
print(f"\nMemory Anchor says:\n{response}")

print("\n" + "=" * 60)
print("TEST 5: 'When do I see the doctor?'")
print("=" * 60)
response, results = ask_about_person("When do I see the doctor?")
print(f"\nMemory Anchor says:\n{response}")


## Step 9: Safety Check — Unknown Person

In [ ]:
unknown_img = Image.new("RGB", (512, 512), (100, 100, 100))
draw = ImageDraw.Draw(unknown_img)
draw.rectangle([100, 100, 400, 400], fill=(50, 50, 50))
draw.text((256, 256), "?", fill="white", anchor="mm")
unknown_path = "./data/raw/unknown_test.jpg"
unknown_img.save(unknown_path)

print("=" * 60)
print("TEST 6: Safety — Unknown person")
print("=" * 60)
results = retrieve_by_image(unknown_path)
top_distance = results["distances"][0][0] if results["distances"][0] else 1.0
top_confidence = max(0, 1 - top_distance)
print(f"Top match confidence: {top_confidence:.0%}")
print(f"Top match distance: {top_distance:.4f}")

if top_confidence < 0.7:
    print("\n[SAFETY] Low confidence — would NOT identify this person.")
    print("Response: I'm not sure who this is. Would you like to tell me about them so I can remember for next time?")
else:
    context = format_context(results)
    response = generate_response(context, "Who is this?")
    print(f"\nResponse: {response}")

Path(unknown_path).unlink()


## Step 10: Adding New Memories (Instant — No Retraining!)

In [ ]:
print("=" * 60)
print("DEMO: Adding a new family member in real-time")
print("=" * 60)

new_person_dir = Path("./data/raw/uncle_joe")
new_person_dir.mkdir(parents=True, exist_ok=True)
img = Image.new("RGB", (512, 512), (180, 200, 180))
draw = ImageDraw.Draw(img)
cx, cy = 256, 226
draw.ellipse([cx-80, cy-80, cx+80, cy+80], fill="white", outline="gray", width=2)
draw.ellipse([cx-30, cy-20, cx-10, cy], fill="gray")
draw.ellipse([cx+10, cy-20, cx+30, cy], fill="gray")
draw.arc([cx-30, cy+10, cx+30, cy+40], 0, 180, fill="gray", width=2)
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 36)
except (IOError, OSError):
    font = ImageFont.load_default()
draw.text((256, 420), "Uncle Joe", fill="black", anchor="mm", font=font)
photo_path = new_person_dir / "photo_1.jpg"
img.save(photo_path)

new_img = Image.open(photo_path).convert("RGB")
new_embedding = clip_model.encode(new_img, convert_to_numpy=True)
new_story = "Uncle Joe is Robert's brother. He was a jazz musician in New Orleans. He taught Arki to play guitar. He always wears a fedora."
new_meta = {"person_name": "Uncle Joe", "relationship": "uncle (husband's brother)", "caption": "Uncle Joe playing jazz at the French Quarter, 2005", "story": new_story, "image_path": str(photo_path.resolve()), "audio_path": ""}
image_collection.upsert(ids=["uncle_joe_img_0"], embeddings=[new_embedding.tolist()], metadatas=[new_meta])
text_embedding = text_model.encode(f"Uncle Joe. {new_meta['caption']}. {new_story}", convert_to_numpy=True)
text_collection.upsert(ids=["uncle_joe_txt_0"], embeddings=[text_embedding.tolist()], metadatas=[new_meta])
print(f"Added Uncle Joe! Total: {image_collection.count()} images, {text_collection.count()} texts")
print(f"With fine-tuning this would require retraining. With RAG: instant.\n")

response, results = ask_about_person("Who played jazz? Tell me about Uncle Joe.")
print(f"Query: 'Who played jazz? Tell me about Uncle Joe.'")
print(f"Memory Anchor says:\n{response}")


## Summary

**What we built:**
- **CLIP** encodes family photos into 512-dim embeddings (runs on CPU)
- **ChromaDB** stores them locally with metadata (names, stories, voice clip paths)
- **Cosine similarity** finds the closest match when shown a new photo
- **Gemma 4** generates warm, grounded responses using only retrieved context
- **Safety**: low-confidence matches trigger a gentle "I don't know" response

**Why RAG wins for dementia care:**
1. **Instant updates** — upload a new photo, immediately searchable
2. **Zero hallucination risk** — every fact comes from stored memories
3. **Runs on a tablet** — CLIP is tiny, ChromaDB is lightweight, Gemma handles generation
4. **Privacy-first** — everything stays local, no cloud, no internet needed
5. **Family can update it** — no ML expertise needed to add new memories

---

*Memory Anchor: Because no one should forget the people who love them.*

**Demo Video:** [Watch on YouTube](https://www.youtube.com/watch?v=PSULKHnq1kY)

**Links:**
- [GitHub (RAG)](https://github.com/Javierg720/memory-anchor-rag) | [GitHub (Fine-tuning)](https://github.com/Javierg720/memory-anchor)
- **Team:** Javier G ([@Javierg720](https://github.com/Javierg720))